In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import kymatio
import os
import glob
import math
import scipy
from loading_data import load_data
from kymatio.numpy import Scattering1D
import matplotlib.ticker as ticker

In [ ]:
def spectra_scattering(data):
    
    T = data.shape[-1]
    J = math.floor(np.log2(T))
    Q = 1

    scattering = Scattering1D(J, T, Q)
    SC = scattering(data)
    meta = scattering.meta()
    return SC, meta

def group_by_order(coefficients: np.ndarray, meta: np.ndarray, order: int=0):
    '''
    group coefficients and corresponding keys from the meta data by order
    order:
        maximum 2
    '''
    idx = np.where(meta['order'] == order)[0].astype(int)
    keys = np.array(meta['key'], dtype=object)
    coefs_keys = keys[idx]
    coefs_order = coefficients[:, idx]
    
    return coefs_order, coefs_keys

def coefficients_normalization(Sn, Sn_lastorder, Sn_keys, order=1):
    '''
    normalize coefficients using Sn/Sn-1
    '''
    if order == 1:
        S1_normalized = Sn / Sn_lastorder
        return S1_normalized
        
    elif order == 2:
        j1_j2 = np.array(list(Sn_keys.flatten()))
        S2_normalized = {}
        S2_normalized['S2'] = {}
        S2_keys = []
        #select second order coefficients and keys by j1
        for j in np.unique(j1_j2[:, 0]):
            idx = np.where(j1_j2[:, 0] == j)[0]
            #S2_z4_grouped[f'{j}'] = S2_z4[:, idx]
            normalized_coefs = Sn[:, idx]/Sn_lastorder[:, j][:, np.newaxis]
            S2_normalized['S2'][f'{j}'] = normalized_coefs
            S2_keys.append(Sn_keys[idx])

        S2_normalized['j1_j2'] = S2_keys

        return S2_normalized
    
def get_normalized_S1_S2(coefficients, meta):
    '''
    Get the grouped and normalized S1 and S2
    '''
    
    S1, S1_keys = group_by_order(coefficients, meta, 1)
    S2, S2_keys = group_by_order(coefficients, meta, 2)
    S0, _ = group_by_order(coefficients, meta, 0)

    S1_normalized = {}
    S2_normalized = {}

    S1_normalized['S1'] = coefficients_normalization(S1, S0, S1_keys, 1)
    S1_normalized['j1'] = S1_keys

    S2_coefficients = coefficients_normalization(S2, S1, S2_keys, 2)
    S2_normalized['S2'] = S2_coefficients['S2']
    S2_normalized['j1_j2'] = S2_coefficients['j1_j2']

    return S1_normalized, S2_normalized

In [ ]:
z=2
data_z2 = load_data(z)
optical_depth_z2 = data_z2.optical_depth()
flux_z2 = data_z2.flux()
nspectra_z2 = optical_depth_z2.shape[0]
print('number of spectra:', flux_z2.shape[0], 'length of spectra:',  flux_z2.shape[1])

In [ ]:
z=3
data_z3 = load_data(z)
optical_depth_z3 = data_z3.optical_depth()
flux_z3 = data_z3.flux()
nspectra_z3 = optical_depth_z3.shape[0]
print('number of spectra:', flux_z3.shape[0], 'length of spectra:',  flux_z3.shape[1])

In [ ]:
#load redshift 4 data
z = 4
data_z4 = load_data(z)
optical_depth_z4 = data_z4.optical_depth()
flux_z4 = data_z4.flux()
nspectra_z4 = optical_depth_z4.shape[0]
print('number of spectra:', flux_z4.shape[0], 'length of spectra:',  flux_z4.shape[1])

In [ ]:
#scattering transform on redshift 2 spectra
coefficients_z2, meta_z2 = spectra_scattering(flux_z2)
coefficients_z2 = coefficients_z2[:, :, 0]
print(coefficients_z2.shape)

In [ ]:
#scattering transform on redshift 3 spectra
coefficients_z3, meta_z3 = spectra_scattering(flux_z3)
coefficients_z3 = coefficients_z3[:, :, 0]
print(coefficients_z3.shape)

In [ ]:
#scattering transform on redshift 4 spectra
coefficients_z4, meta_z4 = spectra_scattering(flux_z4)
coefficients_z4 = coefficients_z4[:, :, 0]
print(coefficients_z4.shape)

In [ ]:
S1_z2, S2_z2 = get_normalized_S1_S2(coefficients_z2, meta_z2)
S1_z2_mean = np.mean(S1_z2['S1'], axis=0)
S2_z2_mean = [np.mean(S2_z2['S2'][f'{j}'], axis=0) for j in range(len(S2_z2['S2']))]

In [ ]:
S1_z3, S2_z3 = get_normalized_S1_S2(coefficients_z3, meta_z3)
S1_z3_mean = np.mean(S1_z3['S1'], axis=0)
S2_z3_mean = [np.mean(S2_z3['S2'][f'{j}'], axis=0) for j in range(len(S2_z3['S2']))]

In [ ]:
S1_z4, S2_z4 = get_normalized_S1_S2(coefficients_z4, meta_z4)
S1_z4_mean = np.mean(S1_z4['S1'], axis=0)
S2_z4_mean = [np.mean(S2_z4['S2'][f'{j}'], axis=0) for j in range(len(S2_z4['S2']))]

In [ ]:
#plot redshift 4 spectra
n = 100
fig, axs = plt.subplots(3, 1, figsize=(18, 12),
                        sharex=True)
axs[0].plot(flux_z4[n], label='z=4')
axs[1].plot(flux_z3[n], label='z=3')
axs[2].plot(flux_z2[n], label='z=2')
axs[0].set_ylabel('flux')
axs[1].set_ylabel('flux')
axs[2].set_ylabel('flux')
axs[2].set_xlabel('velocity/kms')
axs[0].legend(loc='upper right')
axs[1].legend(loc='upper right')
axs[2].legend(loc='upper right')
axs[2].xaxis.set_minor_locator(ticker.MultipleLocator(500))
fig.tight_layout()

In [ ]:
fig, axs = plt.subplots(1, figsize=(20, 6))
j1 = np.array(list(S1_z4['j1'].flatten()))[:, 0]

axs.plot(S1_z4_mean, label='z=4')
axs.plot(S1_z3_mean, label='z=3')
axs.plot(S1_z2_mean, label='z=2')
axs.set_title('First order scattering coefficients', fontsize=14)
axs.set_xticks(j1)
axs.set_xlabel('j1', fontsize=14)
axs.set_ylabel('S1', fontsize=14)
axs.legend()

In [ ]:
#second order scattering plot
fig, axes = plt.subplots(1, len(S2_z4_mean[:-1]), figsize=(20, 7), sharey=True)
fig.subplots_adjust(wspace=0) 
fig.suptitle('Second order scattering coefficients', y=0.93, fontsize=14)
fig.supylabel('S2', x=0.09)

for j, (ax, s2_z4, s2_z3, s2_z2) in enumerate(zip(axes, S2_z4_mean, S2_z3_mean, S2_z2_mean)):
    j2_labels = [e[1] for e in S2_z4['j1_j2'][j]]
    ax.plot(s2_z4, label='z=4')
    ax.plot(s2_z3, label='z=3')
    ax.plot(s2_z2, label='z=2')
    ax.set_xlabel(f'j1={j}, j2')
    ax.set_xticks(range(len(s2_z4)))
    ax.set_xticklabels(j2_labels)
    if ax != axes[0]:
        ax.spines['left'].set_visible(False)
        ax.tick_params(left=False)

handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, bbox_to_anchor=(0.9, 0.85))

#TODO: fix the label